In [63]:
!pip install wikipedia
!pip uninstall -y langchain langchain-core langchain-community langchain-groq
!pip install -U "langchain>=0.3.0" "langchain-community>=0.3.0" "langchain-core>=0.3.0" --force-reinstall
!pip install -U ddgs

Found existing installation: langchain 1.2.8
Uninstalling langchain-1.2.8:
  Successfully uninstalled langchain-1.2.8
Found existing installation: langchain-core 1.2.8
Uninstalling langchain-core-1.2.8:
  Successfully uninstalled langchain-core-1.2.8
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
  Using cached langchain-1.2.8-py3-none-any.whl.metadata (5.0 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_core-1.2.8-py3-none-any.whl.metadata (3.7 kB)
  Using cached langgraph-1.0.7-py3-none-any.whl.metadata (7.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langsmith-0.6.8-py3-none-any.whl.metadata (15 kB)
  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-macosx_11_0_arm64.

In [1]:
from langchain_groq import ChatGroq
api_key = "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"
llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    groq_api_key = api_key
)
response = llm.invoke("What is your name?.")
print(f"{response.content}")

/Users/anirudh/.local/share/virtualenvs/Desktop-sFnGVMJ4/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."


In [2]:

import re
def add_numbers(inputs:str)-> dict:
    """
    Adds a list of numbers in the input directory or extracts numbers from the string
    Parameters:
    - inputs (str):
    string, It should contain numbers that can be extracted and summed.
    Returns:
    - dict: A dictionary with a single key "result" containing the sum of numbers.
    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(x) for x in inputs.replace(",","").split() if x.isdigit()]
    result = sum(numbers)
    return {"result": result}
    

In [3]:
add_numbers("1 2")

{'result': 3}

In [4]:
from langchain_core.tools import Tool
add_tool = Tool(
    name = "Add tool",
    func = add_numbers,
    description = "Adds a list of numbers and returns a result"
)
print("tool object ",add_tool)

tool object  name='Add tool' description='Adds a list of numbers and returns a result' func=<function add_numbers at 0x157fce200>


In [5]:
#Tool name
print(f"Tool name {add_tool.name}")
# Tool description
print(f"Tool description {add_tool.description}")
# Tool function
print(f"Tool function {add_tool.invoke}")

Tool name Add tool
Tool description Adds a list of numbers and returns a result
Tool function <bound method BaseTool.invoke of Tool(name='Add tool', description='Adds a list of numbers and returns a result', func=<function add_numbers at 0x157fce200>)>


In [6]:
test_input = "10 20 30 a b"
print(f"{add_tool.invoke(test_input)}")

{'result': 60}


In [7]:
#Using @tool decorator(recommended way)
from langchain_core.tools import tool
@tool
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input string.
    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.
    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.
    Example Input:
    "Add the numbers 10, 20, and 30."
    Example Output:
    {"result": 60}
    """
    # Use regular expressions to extract all numbers from the input
    #numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]
    
    result = sum(numbers)
    return {"result": result}

In [8]:
print("Name: \n", add_numbers.name)
print("Description: \n", add_numbers.description) 
print("Args: \n", add_numbers.args) 


Name: 
 add_numbers
Description: 
 Adds a list of numbers provided in the input string.
Parameters:
- inputs (str): 
string, it should contain numbers that can be extracted and summed.
Returns:
- dict: A dictionary with a single key "result" containing the sum of the numbers.
Example Input:
"Add the numbers 10, 20, and 30."
Example Output:
{"result": 60}
Args: 
 {'inputs': {'title': 'Inputs', 'type': 'string'}}


In [17]:
test_input = "what is the sum between 10, 20 and 30 " 
print(add_numbers.invoke(test_input))

{'result': 60}


In [9]:
# Comparing the two approaches
print("Tool Constructor Approach:")

print(f"Has Schema: {hasattr(add_tool, 'args_schema')}")
print("\n")

print("@tool Decorator Approach:")


print(f"Has Schema: {hasattr(add_numbers, 'args_schema')}")
print(f"Args Schema Info: {add_numbers.args}")

Tool Constructor Approach:
Has Schema: True


@tool Decorator Approach:
Has Schema: True
Args Schema Info: {'inputs': {'title': 'Inputs', 'type': 'string'}}


In [10]:
from typing import List
@tool
def add_numbers_with_options(numbers:List[float],absolute:bool = False)->float:
    """
    Adds a list of numbers provided as input.
    Parameters:
    -numbers (List[float]): A list of numbers to be summed
    - absolute(bool):If true, use the absolute values of the numbers for summing.
    Returns:
    float: The sum of numbers
    """
    if absolute:
        numbers = [abs(n) for n in numbers]
    return sum(numbers)    

In [11]:
print(f"Args Schema Info: {add_numbers_with_options.args}")
print(f"Args Schema Info: {add_numbers.args}")

Args Schema Info: {'numbers': {'items': {'type': 'number'}, 'title': 'Numbers', 'type': 'array'}, 'absolute': {'default': False, 'title': 'Absolute', 'type': 'boolean'}}
Args Schema Info: {'inputs': {'title': 'Inputs', 'type': 'string'}}


In [12]:
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":False}))
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":True}))

-6.2
6.2


In [13]:
from typing import Dict,Union
@tool
def sum_numbers_with_complex_output(inputs: str) -> Dict[str, Union[float, str]]:
    """
    Extracts and sums all integers and decimal numbers from the input string.

    Parameters:
    - inputs (str): A string that may contain numeric values.

    Returns:
    - dict: A dictionary with the key "result". If numbers are found, the value is their sum (float). 
            If no numbers are found or an error occurs, the value is a corresponding message (str).

    Example Input:
    "Add 10, 20.5, and -3."

    Example Output:
    {"result": 27.5}
    """
    matches = re.findall(r'-?\d+(?:\.\d+)?', inputs)
    if not matches:
        return {"result": "No numbers found in input."}
    try:
        numbers = [float(num) for num in matches]
        total = sum(numbers)
        return {"result": total}
    except Exception as e:
        return {"result": f"Error during summation: {str(e)}"}

In [14]:
@tool
def sum_numbers_from_text(inputs: str) -> float:
    """
    Adds a list of numbers provided in the input string.
    
    Args:
        text: A string containing numbers that should be extracted and summed.
        
    Returns:
        The sum of all numbers found in the input.
    """
    # Use regular expressions to extract all numbers from the input
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    result = sum(numbers)
    return result

In [26]:
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent,initialize_agent

agent = initialize_agent([add_tool], llm, agent="zero-shot-react-description", verbose=True, handle_parsing_errors=True)

/var/folders/kv/9b0l8lv56tsc3yvfy9l9nq180000gn/T/ipykernel_96220/3547703458.py:3: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent([add_tool], llm, agent="zero-shot-react-description", verbose=True, handle_parsing_errors=True)


In [27]:
response =agent.run("In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total.")

/var/folders/kv/9b0l8lv56tsc3yvfy9l9nq180000gn/T/ipykernel_96220/1757704528.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  response =agent.run("In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total.")




> Entering new AgentExecutor chain...
Thought: To find the total GDP of the US, Canada, and Mexico, I need to add their individual GDP values. I can use the Add tool to achieve this.

Action: Add tool
Action Input: 27.72, 2.14, 1.79
Observation: {'result': 0}
Thought:It seems like the Add tool didn't work as expected. Let me try again with a different input format.

Action: Add tool
Action Input: 27.72,2.14,1.79
Observation: {'result': 0}
Thought:It seems like the Add tool still didn't work as expected. Let me try again with the numbers separated by a space, and also consider that the tool might require the numbers to be in a specific format, such as a string with the numbers separated by commas or a list of strings.

Action: Add tool
Action Input: "27.72, 2.14, 1.79"
Observation: {'result': 0}
Thought:It seems like the Add tool still didn't work as expected. Let me try again with the numbers separated by commas, without spaces.

Action: Add tool
Action Input: "27.72,2.14,1.79"
Obser

agent.invoke({"input": "Add 10, 20, two and 30"})

In [28]:
agent.invoke({"input": "Add 10, 20, two and 30"})



> Entering new AgentExecutor chain...
To solve this problem, I first need to convert the word "two" into a numerical value, which is 2. Then, I can use the Add tool to add up all the numbers.

Action: Add tool
Action Input: 10, 20, 2, 30
Observation: {'result': 62}
Thought:Thought: I now know the final answer
Final Answer: 62

> Finished chain.


{'input': 'Add 10, 20, two and 30', 'output': '62'}

In [29]:
!pip install langgraph

In [34]:
from langgraph.prebuilt import create_react_agent
agent_exec = create_react_agent(model= llm ,tools= [sum_numbers_from_text])
msgs = agent_exec.invoke({"messages":["human" ,"Add the numbers 10,20,-30,40"]})
print(msgs["messages"][-1].content)

/var/folders/kv/9b0l8lv56tsc3yvfy9l9nq180000gn/T/ipykernel_96220/4066166914.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_exec = create_react_agent(model= llm ,tools= [sum_numbers_from_text])


The sum of the numbers 10, 20, -30, and 40 is 40.


In [40]:
@tool
def subtract_numbers(inputs:str)-> dict:
    """
    Extracts the numbers from the string and subtracts them.
    This function takes an input in string format,where numbers are seperated by space,comma,or other delimiters.
    It parses the string, extracts valid numeric values, 
    and performs a step-by-step subtraction operation starting with the first number negated.
    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string may include spaces, commas, or 
      other delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Input:
    "100, 20, 10"

    Example Output:
    {"result": -130}

    Notes:
    - Non-numeric characters in the input are ignored.
    - If the input string contains only one valid number, the result will be that number negated.
    - Handles a variety of delimiters (e.g., spaces, commas) but does not validate input formats 
      beyond extracting numeric values.
    """
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit() ]
    if not numbers:
        return {"result":0}
    result = -1 * numbers[0]
    for num in numbers[1:]:
        result -= num
    return {"result":result}    

In [41]:
print("Name: \n", subtract_numbers.name)
print("Description: \n", subtract_numbers.description) 
print("Args: \n", subtract_numbers.args) 

Name: 
 subtract_numbers
Description: 
 Extracts the numbers from the string and subtracts them.
This function takes an input in string format,where numbers are seperated by space,comma,or other delimiters.
It parses the string, extracts valid numeric values, 
and performs a step-by-step subtraction operation starting with the first number negated.
Parameters:
- inputs (str): 
  A string containing numbers to subtract. The string may include spaces, commas, or 
  other delimiters between the numbers.

Returns:
- dict: 
  A dictionary containing the key "result" with the calculated difference as its value. 
  If no valid numbers are found in the input string, the result defaults to 0.

Example Input:
"100, 20, 10"

Example Output:
{"result": -130}

Notes:
- Non-numeric characters in the input are ignored.
- If the input string contains only one valid number, the result will be that number negated.
- Handles a variety of delimiters (e.g., spaces, commas) but does not validate input forma

In [42]:
print("Calling Tool Function:")
test_input = "10 20 30 and four a b" 
print(subtract_numbers.invoke(test_input))  # Example

Calling Tool Function:
{'result': -60}


In [43]:
# Multiplication Tool
@tool
def multiply_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates their product.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the product of the numbers.

    Example Input:
    "2, 3, 4"

    Example Output:
    {"result": 24}

    Notes:
    - If no numbers are found, the result defaults to 1 (neutral element for multiplication).
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]
    print(numbers)

    # If no numbers are found, return 1
    if not numbers:
        return {"result": 1}

    # Calculate the product of the numbers
    result = 1
    for num in numbers:
        result *= num
        print(num)

    return {"result": result}

In [44]:
# Division Tool
@tool
def divide_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates the result of dividing the first number 
    by the subsequent numbers in sequence.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the quotient.

    Example Input:
    "100, 5, 2"

    Example Output:
    {"result": 10.0}

    Notes:
    - If no numbers are found, the result defaults to 0.
    - Division by zero will raise an error.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]


    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Calculate the result of dividing the first number by subsequent numbers
    result = numbers[0]
    for num in numbers[1:]:
        result /= num

    return {"result": result}

In [45]:
# Testing multiply_tool
multiply_test_input = "2, 3, and four "
multiply_result = multiply_numbers.invoke(multiply_test_input)
print("--- Testing MultiplyTool ---")
print(f"Input: {multiply_test_input}")
print(f"Output: {multiply_result}")

[2, 3]
2
3
--- Testing MultiplyTool ---
Input: 2, 3, and four 
Output: {'result': 6}


In [46]:
# Testing divide_tool
divide_test_input = "100, 5, two"
divide_result = divide_numbers.invoke(divide_test_input)
print("--- Testing DivideTool ---")
print(f"Input: {divide_test_input}")
print(f"Output: {divide_result}")

--- Testing DivideTool ---
Input: 100, 5, two
Output: {'result': 20.0}


In [47]:
tools = [add_numbers,subtract_numbers, multiply_numbers, divide_numbers]
tools

[StructuredTool(name='add_numbers', description='Adds a list of numbers provided in the input string.\nParameters:\n- inputs (str): \nstring, it should contain numbers that can be extracted and summed.\nReturns:\n- dict: A dictionary with a single key "result" containing the sum of the numbers.\nExample Input:\n"Add the numbers 10, 20, and 30."\nExample Output:\n{"result": 60}', args_schema=<class 'langchain_core.utils.pydantic.add_numbers'>, func=<function add_numbers at 0x157faf4c0>),
 StructuredTool(name='subtract_numbers', description='Extracts the numbers from the string and subtracts them.\nThis function takes an input in string format,where numbers are seperated by space,comma,or other delimiters.\nIt parses the string, extracts valid numeric values, \nand performs a step-by-step subtraction operation starting with the first number negated.\nParameters:\n- inputs (str): \n  A string containing numbers to subtract. The string may include spaces, commas, or \n  other delimiters be

In [48]:
math_agent = create_react_agent(model = llm ,tools= tools)
result = math_agent.invoke({"messages":["human","What is 25 divided by 5"]})
print(result["messages"][-1].content)

/var/folders/kv/9b0l8lv56tsc3yvfy9l9nq180000gn/T/ipykernel_96220/1410996795.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  math_agent = create_react_agent(model = llm ,tools= tools)


The result of 25 divided by 5 is 5.0.


In [52]:
response_2 = math_agent.invoke({
    "messages": [("human", "Subtract 100, 20, and 10.")]
})

# Get the final answer
final_answer_2 = response_2["messages"][-1].content
print(final_answer_2)

The result of subtracting 100, 20, and 10 is -130.


In [53]:
from langchain_community.utilities import WikipediaAPIWrapper
@tool
def search_wikipedia(query:str)->str:
    """
    Searches wikipedia for information about a given topic
    Parameters:
    -query: It is of string format which is the topic which has to be searched in wikipedia
    Returns:
     str-A summary of the relavant information in Wikipedia
    """
    wiki = WikipediaAPIWrapper()
    return wiki.run(query)

In [56]:
search_wikipedia.invoke("Artificial Intelligence")

'Page: Artificial intelligence\nSummary: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.\nHigh-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as AI: "A lot of cutting edge AI has filtered into gener

In [64]:
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

@tool
def search_google(query:str)->str:
    """
    This function takes a query and searches google for relavant information regarding the topic.
    Parameters:
    -query(str): This is the input on which google performs the search
    Returns:
    -str : It returns the relavant information in string format
    """
    google = DuckDuckGoSearchAPIWrapper()
    return google.run(query)

In [65]:
search_google.invoke("Today's breaking news")

"Read the latest headlines, breaking news , and videos at APNews.com, the definitive source for independent journalism from every corner of the globe. View the latest news and breaking news today for U.S., world, weather, entertainment, politics and health at CNN.com.Catch up on today ' s global news . The latest breaking news , comment and features from The Independent.For free real time breaking news alerts sent straight to your inbox sign up to our breaking news emails. With 1,700 journalists reporting from more than 150 countries, we provide live updates, investigations, photos and video of international and regional news , politics, business, technology... Global News Podcast. BBC on frontline of Colombia' s drugs crackdown.John Virgo, who co-presented TV show Big Break and was a popular BBC commentator for decades, has died."

In [ ]:
from langchain_community.utilities import 